# 抽出型 QA モデルをファインチューニングする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/23_qa/qa_finetune.ipynb)

NLP クックブック 23-05 ／ 質問応答

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

自分の分野の文書に対する抽出型 QA モデルを、質問・文書・答えの位置のデータから学習したい。

In [1]:
!pip install -q datasets accelerate

In [2]:
from datasets import load_dataset
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

ds = load_dataset("sbintuitions/JSQuAD")
train = ds["train"].shuffle(seed=0).select(range(8000))      # 短時間で終わるように一部だけ使う (全体は約 6.3 万件)
valid = ds["validation"].select(range(1000))

model_name = "xlm-roberta-base"                               # 文字位置 (offset mapping) を返せるトークナイザを持つモデル
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
MAX_LEN, STRIDE = 384, 128

In [3]:
def prepare_train(batch):
    """長い文書は重なりを持たせて複数の窓に分け、各窓について答えの開始・終了トークン位置を求める"""
    enc = tokenizer(batch["question"], batch["context"], truncation="only_second", max_length=MAX_LEN, stride=STRIDE,
                    return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length")
    starts, ends = [], []
    for i, offsets in enumerate(enc["offset_mapping"]):
        answers = batch["answers"][enc["overflow_to_sample_mapping"][i]]
        a_start = answers["answer_start"][0]
        a_end = a_start + len(answers["text"][0])
        seq = enc.sequence_ids(i)
        ctx_start, ctx_end = seq.index(1), len(seq) - 1 - seq[::-1].index(1)
        if offsets[ctx_start][0] > a_start or offsets[ctx_end][1] < a_end:      # この窓に答えが入っていない
            starts.append(0); ends.append(0)
        else:
            s = ctx_start
            while s <= ctx_end and offsets[s][0] <= a_start:
                s += 1
            e = ctx_end
            while e >= ctx_start and offsets[e][1] >= a_end:
                e -= 1
            starts.append(s - 1); ends.append(e + 1)
    enc["start_positions"], enc["end_positions"] = starts, ends
    enc.pop("offset_mapping"); enc.pop("overflow_to_sample_mapping")
    return enc

def prepare_valid(batch):
    enc = tokenizer(batch["question"], batch["context"], truncation="only_second", max_length=MAX_LEN, stride=STRIDE,
                    return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length")
    enc["example_id"] = [batch["id"][j] for j in enc["overflow_to_sample_mapping"]]
    enc["offset_mapping"] = [[o if seq == 1 else None for o, seq in zip(offsets, enc.sequence_ids(i))]     # 文書以外の位置は None にしておく
                             for i, offsets in enumerate(enc["offset_mapping"])]
    enc.pop("overflow_to_sample_mapping")
    return enc

train_enc = train.map(prepare_train, batched=True, remove_columns=train.column_names)
valid_enc = valid.map(prepare_valid, batched=True, remove_columns=valid.column_names)
print(len(train), "→", len(train_enc), "窓 /", len(valid), "→", len(valid_enc), "窓")

8000 → 8023 窓 / 1000 → 1004 窓


In [4]:
from transformers import Trainer, TrainingArguments, default_data_collator

args = TrainingArguments(output_dir="qa-out", num_train_epochs=2, learning_rate=3e-5, warmup_steps=100, weight_decay=0.01,
                         per_device_train_batch_size=16, per_device_eval_batch_size=32, fp16=True,
                         logging_steps=100, save_strategy="no", report_to="none", seed=0)
trainer = Trainer(model=model, args=args, train_dataset=train_enc, data_collator=default_data_collator)
trainer.train()

Step,Training Loss
100,4.449693
200,1.940042
300,1.401266
400,1.189796
500,1.115769
600,0.793967
700,0.833405
800,0.796467
900,0.818769
1000,0.747864


TrainOutput(global_step=1004, training_loss=1.4076238998853827, metrics={'train_runtime': 198.4198, 'train_samples_per_second': 80.869, 'train_steps_per_second': 5.06, 'total_flos': 3144575818939392.0, 'train_loss': 1.4076238998853827, 'epoch': 2.0})

In [5]:
import collections
import re
import unicodedata
import numpy as np

def postprocess(examples, features, start_logits, end_logits, n_best=20, max_answer_len=30):
    """窓ごとの予測を質問単位にまとめ、スコアが最大の区間を答えとする"""
    by_example = collections.defaultdict(list)
    for i, f in enumerate(features):
        by_example[f["example_id"]].append(i)
    predictions = {}
    for ex in examples:
        best = (-1e9, "")
        for i in by_example[ex["id"]]:
            offsets = features[i]["offset_mapping"]
            for s in np.argsort(start_logits[i])[::-1][:n_best]:
                for e in np.argsort(end_logits[i])[::-1][:n_best]:
                    if offsets[s] is None or offsets[e] is None or e < s or e - s + 1 > max_answer_len:
                        continue
                    score = start_logits[i][s] + end_logits[i][e]
                    if score > best[0]:
                        best = (score, ex["context"][offsets[s][0]:offsets[e][1]])
        predictions[ex["id"]] = best[1]
    return predictions

norm = lambda s: re.sub(r"[\s。、．，,.!?！？「」『』()（）]", "", unicodedata.normalize("NFKC", s).lower())

def char_f1(pred, gold):
    p, g = list(norm(pred)), list(norm(gold))
    common = sum((collections.Counter(p) & collections.Counter(g)).values())
    return 2 * common / (len(p) + len(g)) if common else 0.0

out = trainer.predict(valid_enc.remove_columns(["example_id", "offset_mapping"]))
preds = postprocess(valid, valid_enc, *out.predictions)
em = np.mean([any(norm(preds[ex["id"]]) == norm(a) for a in ex["answers"]["text"]) for ex in valid])
f1 = np.mean([max(char_f1(preds[ex["id"]], a) for a in ex["answers"]["text"]) for ex in valid])
print(f"EM = {em:.3f}  F1 (文字) = {f1:.3f}")
for ex in valid.select(range(3)):
    print(ex["question"], "→", preds[ex["id"]], "/ 正解:", ex["answers"]["text"][0])

EM = 0.750  F1 (文字) = 0.883
日本で梅雨がないのは北海道とどこか。 → 小笠原諸島 / 正解: 小笠原諸島
梅雨とは何季の一種か? → 雨季 / 正解: 雨季
梅雨は、世界的にどのあたりで見られる気象ですか？ → 北海道と小笠原諸島を除く日本、朝鮮半島南部 / 正解: 東アジアの広範囲


- 抽出型 QA は「答えの開始位置」と「終了位置」をそれぞれトークン分類として学習する。前処理の中心は、文字位置で与えられた答えをトークン位置に直すこと。
- 長い文書は、重なり (`stride`) を持たせた複数の窓に分ける。答えが入っていない窓は、先頭の特殊トークンを正解にする。
- 予測は窓単位で出るので、同じ質問の窓をまとめて最良の区間を選ぶ後処理が要る。開始 > 終了や長すぎる区間はここで除く。
- T4 GPU で 8000 件・2 エポックが 10 分程度。この 8000 件では EM が 0.75 前後、文字 F1 が 0.88 前後。全データ（約 6.3 万件）で学習すると EM は 0.82、文字 F1 は 0.94 程度まで上がるが、学習には 5 倍以上の時間がかかる。日本語専用の BERT のほうが精度は高いが、文字位置を返さないトークナイザだと位置合わせを自分で実装する必要がある。